<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/IMG2VID-WAN22-A14B(faster).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install ComfyUI, SageAttention, check GPU, and download Native FP8 Wan 2.2 I2V A14B
import os, json, subprocess

os.makedirs("/content", exist_ok=True)
%cd /content

FAMILY = "a14b"

# Clipboard-safe repository & model URLs
COMFY_REPO = "/".join(["https:", "", "github.com", "comfyanonymous", "ComfyUI.git"])
MMAUDIO_REPO = "/".join(["https:", "", "github.com", "hkchengrex", "MMAudio.git"])
HF_WAN = "/".join(["https:", "", "huggingface.co", "Comfy-Org", "Wan-2.2", "resolve", "main", "split_files"])

# Clone or update ComfyUI safely (preserves existing /models folder)
os.makedirs("/content/ComfyUI", exist_ok=True)
subprocess.run(["git", "-C", "/content/ComfyUI", "init", "-q"], check=True)
subprocess.run(["git", "-C", "/content/ComfyUI", "remote", "remove", "origin"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "-C", "/content/ComfyUI", "remote", "add", "origin", COMFY_REPO], check=True)
subprocess.run(["git", "-C", "/content/ComfyUI", "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", "/content/ComfyUI", "checkout", "-f", "origin/master"], check=True)

# Install dependencies into the active Jupyter Python environment
%pip install -q -r /content/ComfyUI/requirements.txt
%pip install -q fastapi uvicorn requests psutil sageattention
!apt-get update -qq && apt-get install -qqy aria2 ffmpeg > /dev/null

# Local user LoRAs folder
USER_LORAS = "/content/ComfyUI/models/loras/user"
os.makedirs(USER_LORAS, exist_ok=True)

# GPU Check
try:
    gpu_lines = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    ).stdout.strip().split("\n")
    gpu = gpu_lines[0] if gpu_lines else ""
except FileNotFoundError:
    gpu = ""

if "," not in gpu:
    raise RuntimeError("No GPU detected via nvidia-smi.")
gpu_name, vram_mib = [x.strip() for x in gpu.rsplit(",", 1)]
GPU = {"name": gpu_name, "vram_gib": round(int(vram_mib) / 1024, 1)}
if GPU["vram_gib"] < 20:
    raise RuntimeError(f"GPU ({GPU['name']}, {GPU['vram_gib']} GB) is too small for A14B. Use L4 or A100.")

# Native ComfyUI FP8 Scaled Models
FILES = {
    "high": ("wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors", f"{HF_WAN}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors", "diffusion_models"),
    "low": ("wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors", f"{HF_WAN}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors", "diffusion_models"),
    "lora_high": ("wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", f"{HF_WAN}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "loras"),
    "lora_low": ("wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", f"{HF_WAN}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", "loras"),
    "text_encoder": ("umt5_xxl_fp8_e4m3fn_scaled.safetensors", f"{HF_WAN}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders"),
    "vae": ("wan_2.1_vae.safetensors", f"{HF_WAN}/vae/wan_2.1_vae.safetensors", "vae"),
}
DEFAULTS = {
    "fps": 16, "max_frames": 121, "seconds": 5, "size": "480p",
    "fast": {"steps": 4, "cfg": 1.0, "shift": 5.0, "sampler": "euler", "scheduler": "simple"},
    "quality": {"steps": 12, "cfg": 4.0, "shift": 7.0, "sampler": "uni_pc", "scheduler": "beta"},
    "balanced": {"steps": 8, "cfg": 4.0, "shift": 5.0, "sampler": "euler", "scheduler": "simple"},
    "default_mode": "balanced",
}
LABEL = "Wan 2.2 I2V A14B (Native FP8 + SageAttention)"

# Optional Sound (MMAudio) - set ADD_SOUND = False to skip ~8 GB download
ADD_SOUND = True
SOUND_OK = False
if ADD_SOUND:
    print("Installing MMAudio (sound for videos)...")
    if not os.path.isdir("/content/MMAudio/.git"):
        subprocess.run(["git", "clone", "-q", MMAUDIO_REPO, "/content/MMAudio"], check=True)
    subprocess.run(["git", "-C", "/content/MMAudio", "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", "/content/MMAudio", "checkout", "-q", "974010a"], check=True)
    %pip install -q torchdiffeq open_clip_torch timm av einops librosa omegaconf colorlog
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("apple/DFN5B-CLIP-ViT-H-14-384", allow_patterns=["open_clip*"])
        snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", allow_patterns=["config.json", "bigvgan_generator.pt"])
        subprocess.run(["python", "-c", "from mmaudio.eval_utils import all_model_cfg as c; c['large_44k_v2'].download_if_needed()"],
                       cwd="/content/MMAudio", check=True)
        SOUND_OK = True
    except Exception as e:
        print(f"Sound turned off: MMAudio setup failed ({e}). Silent videos still work.")

# Download models (resumable via aria2c -c) and write video_config.json
for key, (name, url, folder) in FILES.items():
    print(f"Downloading {name}...")
    os.makedirs(f"/content/ComfyUI/models/{folder}", exist_ok=True)
    subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url,
                    "-d", f"/content/ComfyUI/models/{folder}", "-o", name], check=True)

json.dump({
    "family": FAMILY, "label": LABEL, "gpu": GPU, "defaults": DEFAULTS, "sound": SOUND_OK,
    "files": {k: v[0] for k, v in FILES.items()}
}, open("/content/video_config.json", "w"), indent=1)
print(f"READY FOR CELL 2 -> GPU: {GPU['name']} ({GPU['vram_gib']} GB) | {LABEL} | sound {'on' if SOUND_OK else 'off'}")


In [ ]:
# Cell 2: Build /content/video_server.py with Native FP8 + SageAttention
import os, glob, json, subprocess, py_compile

if os.path.exists("/content/video_server.log"):
    os.remove("/content/video_server.log")

# Fetch upstream FastAPI server template if not already cached
if not os.path.isdir("/tmp/up_repo"):
    repo = "/".join(["https:", "", "github.com", "niofrequency", "google-collab_image.git"])
    subprocess.run(["git", "clone", "-q", repo, "/tmp/up_repo"], check=True)

found_code = None
for path in sorted(glob.glob("/tmp/up_repo/*.ipynb")):
    nb = json.load(open(path, encoding="utf-8"))
    for c in nb["cells"]:
        src = "".join(c["source"])
        if ("FastAPI" in src or "uvicorn" in src) and "nodes_wan" in src and "Supervisor" not in src:
            found_code = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith(("!", "%")))
            break
    if found_code:
        break

assert found_code, "Could not locate Wan 2.2 FastAPI server template!"

# Replace GGUF loader block with Native FP8 + SageAttention
start_marker = 'logging.getLogger("root")'
end_marker = 'print("Models loaded.", flush=True)'

new_fp8_block = '''logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import comfy.cli_args
comfy.cli_args.args.use_sage_attention = True

import nodes
import comfy.utils
import comfy.model_management
from comfy_extras import nodes_wan, nodes_model_advanced

M = nodes.NODE_CLASS_MAPPINGS
UNETLoader, CLIPLoader = M["UNETLoader"](), M["CLIPLoader"]()
VAELoader, LoraLoaderModelOnly = M["VAELoader"](), M["LoraLoaderModelOnly"]()
CLIPTextEncode, VAEDecode = M["CLIPTextEncode"](), M["VAEDecode"]()
KSampler, KSamplerAdvanced = M["KSampler"](), M["KSamplerAdvanced"]()
ModelSamplingSD3 = nodes_model_advanced.ModelSamplingSD3()
F = {k: (v[0] if isinstance(v, (list, tuple)) else v) for k, v in CONFIG["files"].items()}
D = CONFIG["defaults"]

print(f"Loading {CONFIG['label']}...", flush=True)
with torch.inference_mode():
    clip = CLIPLoader.load_clip(F["text_encoder"], "wan")[0]
    vae = VAELoader.load_vae(F["vae"])[0]
    if FAMILY == "a14b":
        weight_dtype = "fp8_e4m3fn_fast" if torch.cuda.get_device_capability() == (8, 9) else "default"
        base_high = UNETLoader.load_unet(F["high"], weight_dtype)[0]
        base_low = UNETLoader.load_unet(F["low"], weight_dtype)[0]
        fast_high = LoraLoaderModelOnly.load_lora_model_only(base_high, F["lora_high"], 1.0)[0]
        fast_low = LoraLoaderModelOnly.load_lora_model_only(base_low, F["lora_low"], 1.0)[0]
        MODELS = {
            "fast": tuple(ModelSamplingSD3.patch(m, D["fast"]["shift"])[0] for m in (fast_high, fast_low)),
            "quality": tuple(ModelSamplingSD3.patch(m, D["quality"]["shift"])[0] for m in (base_high, base_low)),
            "balanced": tuple(ModelSamplingSD3.patch(m, D["balanced"]["shift"])[0] for m in (base_high, fast_high, fast_low)),
        }
    else:
        base = UNETLoader.load_unet(F["model"], "default")[0]
        fast = LoraLoaderModelOnly.load_lora_model_only(base, F["lora_turbo"], 1.0)[0]
        MODELS = {
            "fast": ModelSamplingSD3.patch(fast, D["fast"]["shift"])[0],
            "quality": ModelSamplingSD3.patch(base, D["quality"]["shift"])[0],
        }
print("Models loaded.", flush=True)'''

assert start_marker in found_code and end_marker in found_code, "Model markers not found!"
i0 = found_code.index(start_marker)
i1 = found_code.index(end_marker) + len(end_marker)
final_code = found_code[:i0] + new_fp8_block + found_code[i1:]

open("/content/video_server.py", "w", encoding="utf-8").write(final_code)
py_compile.compile("/content/video_server.py", doraise=True)
print(f"READY FOR CELL 3 -> Wrote & verified /content/video_server.py ({len(final_code.splitlines())} lines).")


In [ ]:
# Cell 3: Start the generation server and Cloudflare tunnel
import os, re, sys, time, subprocess, threading, urllib.request

SERVER_SCRIPT = "/content/video_server.py"
SERVER_LOG = "/content/video_server.log"
CRASH_FILE = "/content/server_crash.txt"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"

if not os.path.exists(SERVER_SCRIPT):
    raise RuntimeError(f"{SERVER_SCRIPT} does not exist! Run Cell 2 first.")

server_text = open(SERVER_SCRIPT, errors="replace").read()
if "Supervisor" in server_text or ("FastAPI" not in server_text and "uvicorn" not in server_text):
    raise RuntimeError(f"{SERVER_SCRIPT} is invalid! Re-run Cell 2 first.")

def server_up():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3).status == 200
    except Exception:
        return False

def log_tail(n=25):
    try:
        return "".join(open(SERVER_LOG, errors="replace").readlines()[-n:])
    except OSError:
        return ""

class Supervisor:
    def __init__(self):
        self.proc, self.thread, self.stopping, self.gave_up, self.crashes = None, None, False, False, []

    def _loop(self):
        while not self.stopping:
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n===== {time.strftime('%H:%M:%S')} starting the server =====\n")
                log.flush()
                self.proc = subprocess.Popen(
                    [sys.executable, "-u", SERVER_SCRIPT],
                    cwd="/content/ComfyUI",
                    stdout=log,
                    stderr=subprocess.STDOUT,
                    env={**os.environ, "PYTHONUNBUFFERED": "1"},
                    start_new_session=True,
                )
                code = self.proc.wait()
            if self.stopping:
                break
            why = "ran out of memory" if code in (-9, 137) else f"crashed (exit code {code})"
            self.crashes = [t for t in self.crashes if time.time() - t < 600] + [time.time()]
            if len(self.crashes) >= 4:
                msg = f"The server {why} 4 times in 10 minutes. Check Cell 4 logs, then re-run Cell 3."
                self.gave_up = True
            else:
                msg = f"The server {why} at {time.strftime('%H:%M')} and restarted."
            with open(CRASH_FILE, "w") as fh:
                fh.write(msg)
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n!!!!! {msg}\n")
            if self.gave_up:
                break
            time.sleep(3)

    def start(self):
        self.stopping, self.gave_up = False, False
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.stopping = True
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(30)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        if self.thread:
            self.thread.join(35)

if "SUPERVISOR" in globals():
    print("Restarting server...")
    try:
        SUPERVISOR.stop()
    except Exception:
        pass
if os.path.exists(CRASH_FILE):
    os.remove(CRASH_FILE)
SUPERVISOR = Supervisor()
SUPERVISOR.start()

print("Loading video models into VRAM (usually 30-60s on A100)...")
t0 = time.time()
while not server_up():
    if SUPERVISOR.gave_up or time.time() - t0 > 900:
        print("Server failed to start. Last log lines:\n" + log_tail(30))
        raise RuntimeError("Server failed to start — see log above")
    time.sleep(5)
print(f"Server is up after {time.time() - t0:.0f}s.")

if globals().get("TUNNEL_URL") and (globals().get("TUNNEL_PROC") is None or TUNNEL_PROC.poll() is not None):
    TUNNEL_URL = None
if not globals().get("TUNNEL_URL"):
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    if not os.path.exists(CLOUDFLARED):
        cf_url = "/".join([
            "https:", "", "github.com", "cloudflare", "cloudflared",
            "releases", "latest", "download", "cloudflared-linux-amd64"
        ])
        subprocess.run(["wget", "-q", "-O", CLOUDFLARED, cf_url], check=True)
        os.chmod(CLOUDFLARED, 0o755)
    TUNNEL_PROC = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    t0 = time.time()
    while time.time() - t0 < 60 and not globals().get("TUNNEL_URL"):
        line = TUNNEL_PROC.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
    threading.Thread(target=lambda: [None for _ in TUNNEL_PROC.stdout], daemon=True).start()

if globals().get("TUNNEL_URL"):
    print("=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("Paste this URL into the website: Image to Video tab -> Video server URL.")
    print("=" * 60)
else:
    print("Could not start Cloudflare tunnel. Run Cell 3 again.")


In [ ]:
# Cell 4 (Optional): Check live GPU memory, tunnel health, and server logs
import time, subprocess

try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"System RAM: {(vm.total - vm.available) / 1e9:.1f} / {vm.total / 1e9:.1f} GB")
except Exception:
    pass

gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if gpu:
    print(f"GPU VRAM: {gpu}")

sup = globals().get("SUPERVISOR")
if sup is not None:
    running = sup.proc is not None and sup.proc.poll() is None
    print("Server status:", "running" if running else ("stopped — run Cell 3" if sup.gave_up else "starting / restarting"))
    print(f"Crashes (last 10m): {len([t for t in sup.crashes if time.time() - t < 600])}")

tp = globals().get("TUNNEL_PROC")
if globals().get("TUNNEL_URL"):
    alive = tp is not None and tp.poll() is None
    print(f"Tunnel URL: {TUNNEL_URL}", "(active)" if alive else "(STOPPED — run Cell 3 again)")

print("-" * 60)
try:
    print("".join(open("/content/video_server.log", errors="replace").readlines()[-40:]))
except OSError:
    print("No log yet — run Cell 3 first.")
